# M1 — Prédiction de panne à 7 jours : comparaison des modèles
Partie 2 (Serveurs) — ticket SCRUM-48. Données : ST12000NM0008, avril 2025 → mars 2026 (12 mois) (sortie de `nettoyage_smart.py`).

In [1]:
import os
import numpy as np
import pandas as pd

df = pd.read_parquet(os.path.expanduser("~/data/backblaze/clean"))
df = df.sort_values(["device_id", "date"]).reset_index(drop=True)

print(df.shape)
print(df["label_7j"].value_counts())

(6716215, 11)
label_7j
0    6713166
1       3049
Name: count, dtype: int64


## 1. Indicateurs sur 7 jours

In [2]:
COMPTEURS = ["smart_5_raw", "smart_187_raw", "smart_197_raw", "smart_198_raw"]
par_disque = df.groupby("device_id", sort=False)

for c in COMPTEURS:
    df[f"{c}_var7"] = (df[c] - par_disque[c].shift(7)).fillna(0)

df["smart_194_moy7"] = (par_disque["smart_194_raw"]
                        .rolling(7, min_periods=1).mean()
                        .reset_index(level=0, drop=True))

VARIABLES = ["smart_5_raw", "smart_9_raw", "smart_187_raw", "smart_194_raw",
             "smart_197_raw", "smart_198_raw",
             *[f"{c}_var7" for c in COMPTEURS], "smart_194_moy7"]
print(len(VARIABLES), "variables")
df[["device_id", "date", "smart_5_raw", "smart_5_raw_var7"]].head(10)

11 variables


,device_id,date,smart_5_raw,smart_5_raw_var7
0,ZHZ07RNQ,2025-04-01,0.0,0.0
1,ZHZ07RNQ,2025-04-02,0.0,0.0
2,ZHZ07RNQ,2025-04-03,0.0,0.0
3,ZHZ07RNQ,2025-04-04,0.0,0.0
4,ZHZ07RNQ,2025-04-05,0.0,0.0
5,ZHZ07RNQ,2025-04-06,0.0,0.0
6,ZHZ07RNQ,2025-04-07,0.0,0.0
7,ZHZ07RNQ,2025-04-08,0.0,0.0
8,ZHZ07RNQ,2025-04-09,0.0,0.0
9,ZHZ07RNQ,2025-04-10,0.0,0.0


In [3]:
print((df["smart_5_raw_var7"] != 0).sum(), "lignes où smart_5 a varié en 7 jours")

disque = df.loc[df["label"] == 1, "device_id"].iloc[0]
colonnes = ["date", "smart_5_raw", "smart_5_raw_var7", "smart_187_raw", "smart_187_raw_var7", "label_7j"]
df.loc[df["device_id"] == disque, colonnes].tail(12)

40382 lignes où smart_5 a varié en 7 jours


,date,smart_5_raw,smart_5_raw_var7,smart_187_raw,smart_187_raw_var7,label_7j
25211,2025-08-09,5576.0,544.0,214.0,1.0,0
25212,2025-08-10,5960.0,904.0,214.0,1.0,0
25213,2025-08-11,6088.0,1016.0,214.0,1.0,0
25214,2025-08-12,6152.0,1032.0,214.0,1.0,0
25215,2025-08-13,6168.0,1016.0,214.0,1.0,0
25216,2025-08-14,6168.0,760.0,214.0,1.0,0
25217,2025-08-15,6168.0,728.0,214.0,0.0,0
25218,2025-08-16,6168.0,592.0,224.0,10.0,0
25219,2025-08-17,6168.0,208.0,226.0,12.0,0
25220,2025-08-18,6168.0,80.0,227.0,13.0,0


In [4]:
VARIATIONS = [f"{c}_var7" for c in COMPTEURS]
(df.groupby("label_7j")[VARIATIONS]
   .apply(lambda g: (g > 0).mean() * 100)
   .round(2))

,smart_5_raw_var7,smart_187_raw_var7,smart_197_raw_var7,smart_198_raw_var7
label_7j,,,,
0,0.58,0.44,0.11,0.11
1,44.11,61.95,25.29,25.29


Les variations sur 7 jours séparent bien mieux les disques que les valeurs brutes : une hausse d'erreurs en 7 jours concerne 24 à 59 % des lignes « panne dans 7 jours », contre moins de 0,6 % des lignes saines (×70 à ×207).

## 2. Découpage par disque et rééquilibrage
Découpage 80 / 20 **par disque** (stratifié sur les disques en panne) : un même disque n'est jamais dans les deux jeux. Le découpage par date n'est pas appliqué en plus : avec 6 mois et 241 pannes, il ne laisserait qu'une dizaine de pannes dans le test.
Rééquilibrage uniquement sur l'entraînement : toutes les lignes positives + 50 lignes saines par ligne positive. Le test garde le déséquilibre réel.

In [5]:
from sklearn.model_selection import train_test_split

GRAINE = 42

disques = df.groupby("device_id")["label"].max().rename("en_panne").reset_index()
d_train, d_test = train_test_split(disques, test_size=0.2,
                                   stratify=disques["en_panne"], random_state=GRAINE)

train = df[df["device_id"].isin(d_train["device_id"])]
test = df[df["device_id"].isin(d_test["device_id"])]

assert set(train["device_id"]).isdisjoint(test["device_id"])
for nom, jeu in [("train", train), ("test", test)]:
    print(f"{nom:5} : {jeu['device_id'].nunique():6} disques, "
          f"{jeu.groupby('device_id')['label'].max().sum():3} en panne, "
          f"{len(jeu):8} lignes, {int(jeu['label_7j'].sum()):4} lignes positives")

train :  15232 disques, 408 en panne,  5373348 lignes, 2457 lignes positives
test  :   3809 disques, 102 en panne,  1342867 lignes,  592 lignes positives


In [6]:
RATIO = 200

positifs = train[train["label_7j"] == 1]
negatifs = train[train["label_7j"] == 0].sample(n=RATIO * len(positifs), random_state=GRAINE)
train_eq = pd.concat([positifs, negatifs]).sample(frac=1, random_state=GRAINE)

X_train, y_train = train_eq[VARIABLES], train_eq["label_7j"]
X_test, y_test = test[VARIABLES], test["label_7j"]

print("Entraînement :", X_train.shape, "| positifs :", int(y_train.sum()))
print("Test         :", X_test.shape, "| positifs :", int(y_test.sum()))

Entraînement : (493857, 11) | positifs : 2457
Test         : (1342867, 11) | positifs : 592


## 3. Comparaison des 4 modèles
Régression logistique (référence), Random Forest, XGBoost (ML) et MLP (DL), évalués sur le jeu de test complet, au seuil de 0,5.

In [7]:
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from xgboost import XGBClassifier


def log_signe(x):
    """Écrase les très grandes valeurs (compteurs, heures) en gardant le signe."""
    return np.sign(x) * np.log1p(np.abs(x))


def normalise(modele):
    return make_pipeline(FunctionTransformer(log_signe), StandardScaler(), modele)


modeles = {
    "Régression logistique": normalise(LogisticRegression(max_iter=1000, random_state=GRAINE)),
    "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                            n_jobs=-1, random_state=GRAINE),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                             n_jobs=-1, random_state=GRAINE),
    "MLP": normalise(MLPClassifier(hidden_layer_sizes=(32, 16), early_stopping=True,
                                   max_iter=300, random_state=GRAINE)),
}

In [8]:
from sklearn.metrics import (average_precision_score, f1_score,
                             precision_score, recall_score)

disques_en_panne = test.loc[y_test == 1, "device_id"].unique()


def evaluer(nom, modele):
    debut = time.time()
    modele.fit(X_train, y_train)
    duree = time.time() - debut

    proba = modele.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    alertes_justes = test.loc[(y_test == 1) & (pred == 1), "device_id"].nunique()
    return {
        "modèle": nom,
        "recall": round(recall_score(y_test, pred), 3),
        "précision": round(precision_score(y_test, pred, zero_division=0), 3),
        "F1": round(f1_score(y_test, pred), 3),
        "PR-AUC": round(average_precision_score(y_test, proba), 3),
        "fausses alertes": int(((pred == 1) & (y_test == 0)).sum()),
        "disques en panne détectés": f"{alertes_justes}/{len(disques_en_panne)}",
        "entraînement (s)": round(duree, 1),
    }


resultats = []
for nom, modele in modeles.items():
    print("Entraînement :", nom)
    resultats.append(evaluer(nom, modele))

comparaison = pd.DataFrame(resultats).set_index("modèle")
comparaison

Entraînement : Régression logistique


/home/imane/maintenance-predictive/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Entraînement : Random Forest
Entraînement : XGBoost
Entraînement : MLP


,recall,précision,F1,PR-AUC,fausses alertes,disques en panne détectés,entraînement (s)
modèle,,,,,,,
Régression logistique,0.382,0.142,0.207,0.116,1367,66/102,1.6
Random Forest,0.404,0.178,0.247,0.127,1106,70/102,50.4
XGBoost,0.353,0.145,0.205,0.140,1236,68/102,3.5
MLP,0.412,0.154,0.224,0.134,1339,75/102,17.8


In [9]:
os.makedirs("../resultats", exist_ok=True)
comparaison.to_csv("../resultats/comparaison_m1.csv")
print("Enregistré : partie2-serveurs/resultats/comparaison_m1.csv")

Enregistré : partie2-serveurs/resultats/comparaison_m1.csv


## 4. Conclusion

**6 mois (ratio 1 pour 50)** : précision ≈ 5 %, F1 ≈ 0,10, PR-AUC 0,05 à 0,10 — trop de fausses alertes.

**12 mois (ratio 1 pour 200)** : précision 14 à 18 %, F1 0,21 à 0,25, PR-AUC 0,12 à 0,14. Doubler les pannes (241 → 511) a triplé la précision ; les modèles à arbres rattrapent le MLP.

Les 4 modèles sont au même niveau (écarts de ±0,02 du même ordre que la variabilité du MLP). **Modèle retenu : XGBoost (1 pour 200)** — meilleure PR-AUC (0,146), stable, rapide, compatible avec SHAP TreeExplainer.

Au seuil 0,8 : environ 2 pannes sur 3 anticipées pour ≈ 3 fausses alertes par jour sur 3 800 disques. Le seuil final sera fixé en SCRUM-49.

## 5. Réduire les fausses alertes
Pistes testées : garder plus de disques sains à l'entraînement, monter le seuil d'alerte, exiger plusieurs jours d'alerte consécutifs.

In [10]:
from sklearn.base import clone


def entrainer(modele, ratio):
    negatifs = train[train["label_7j"] == 0]
    n = min(len(negatifs), ratio * len(positifs))
    jeu = pd.concat([positifs, negatifs.sample(n=n, random_state=GRAINE)])
    modele = clone(modele).fit(jeu[VARIABLES], jeu["label_7j"])
    return modele, modele.predict_proba(X_test)[:, 1]


lignes, probas = [], {}
for ratio in [50, 200, 500]:
    for nom, modele in modeles.items():
        debut = time.time()
        _, proba = entrainer(modele, ratio)
        probas[(nom, ratio)] = proba
        lignes.append({"modèle": nom, "ratio": ratio,
                       "PR-AUC": round(average_precision_score(y_test, proba), 3),
                       "temps (s)": round(time.time() - debut, 1)})
        print(lignes[-1])

pd.DataFrame(lignes).pivot(index="modèle", columns="ratio", values="PR-AUC")

/home/imane/maintenance-predictive/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


{'modèle': 'Régression logistique', 'ratio': 50, 'PR-AUC': np.float64(0.116), 'temps (s)': 4.8}
{'modèle': 'Random Forest', 'ratio': 50, 'PR-AUC': np.float64(0.129), 'temps (s)': 10.4}
{'modèle': 'XGBoost', 'ratio': 50, 'PR-AUC': np.float64(0.127), 'temps (s)': 2.0}
{'modèle': 'MLP', 'ratio': 50, 'PR-AUC': np.float64(0.139), 'temps (s)': 9.0}


/home/imane/maintenance-predictive/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


{'modèle': 'Régression logistique', 'ratio': 200, 'PR-AUC': np.float64(0.116), 'temps (s)': 2.3}
{'modèle': 'Random Forest', 'ratio': 200, 'PR-AUC': np.float64(0.124), 'temps (s)': 39.9}
{'modèle': 'XGBoost', 'ratio': 200, 'PR-AUC': np.float64(0.146), 'temps (s)': 4.3}
{'modèle': 'MLP', 'ratio': 200, 'PR-AUC': np.float64(0.115), 'temps (s)': 18.2}


/home/imane/maintenance-predictive/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


{'modèle': 'Régression logistique', 'ratio': 500, 'PR-AUC': np.float64(0.115), 'temps (s)': 3.4}
{'modèle': 'Random Forest', 'ratio': 500, 'PR-AUC': np.float64(0.12), 'temps (s)': 3412.9}
{'modèle': 'XGBoost', 'ratio': 500, 'PR-AUC': np.float64(0.117), 'temps (s)': 14.8}
{'modèle': 'MLP', 'ratio': 500, 'PR-AUC': np.float64(0.14), 'temps (s)': 40.4}


ratio,50,200,500
modèle,,,
MLP,0.139,0.115,0.140
Random Forest,0.129,0.124,0.120
Régression logistique,0.116,0.116,0.115
XGBoost,0.127,0.146,0.117


In [12]:
nb_jours = test["date"].nunique()
nb_pannes = test.loc[y_test == 1, "device_id"].nunique()


def bilan(proba, seuil, jours=1):
    alerte = pd.Series((proba >= seuil).astype(int), index=test.index)
    if jours > 1:
        alerte = (alerte.groupby(test["device_id"]).rolling(jours).sum()
                        .reset_index(level=0, drop=True) >= jours).astype(int)
    alerte = alerte.astype(bool)
    justes = alerte & (y_test == 1)
    detectes = test.loc[justes, "device_id"].nunique()
    return {"seuil": seuil, "jours consécutifs": jours,
            "disques détectés": f"{detectes}/{nb_pannes}",
            "taux de détection": round(detectes / nb_pannes, 2),
            "fausses alertes / jour": round((alerte & (y_test == 0)).sum() / nb_jours, 1),
            "précision": round(justes.sum() / max(alerte.sum(), 1), 3)}


meilleur = ("XGBoost", 200)   # à remplacer par le meilleur couple (modèle, ratio) de la cellule A
proba = probas[meilleur]
pd.DataFrame([bilan(proba, s, j) for j in [1, 2]
              for s in [0.5, 0.7, 0.8, 0.9, 0.95, 0.98]])

,seuil,jours consécutifs,disques détectés,taux de détection,fausses alertes / jour,précision
0,0.50,1,66/102,0.65,3.6,0.142
1,0.70,1,41/102,0.40,1.7,0.162
2,0.80,1,34/102,0.33,0.9,0.202
3,0.90,1,20/102,0.20,0.3,0.348
4,0.95,1,11/102,0.11,0.1,0.588
5,0.98,1,4/102,0.04,0.0,0.800
6,0.50,2,49/102,0.48,3.0,0.134
7,0.70,2,30/102,0.29,1.3,0.151
8,0.80,2,21/102,0.21,0.7,0.190
9,0.90,2,13/102,0.13,0.2,0.360
